In [1]:
# %% 셀 1: 데이터 로드 + 임베딩 로드 + STT 로드 + train/eval 분리
import os, json, random
import numpy as np
import torch
import polars as pl
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

POS_DIR = "./data/8_telop_position"
STT_DIR = "./data/4_stt_results"
EMB_PATH = "./data/8_text_embeddings.pt"
GRID_W = 80
GRID_H = 80
CELL_SIZE_X = 9
CELL_SIZE_Y = 16
EVAL_PER_CHANNEL = 5
SEED = 42
NUM_WORKERS = 32
FPS = 10

# ── 임베딩 로드 ──
text2emb = torch.load(EMB_PATH, weights_only=True)
EMB_DIM = next(iter(text2emb.values())).shape[0]
ZERO_EMB = torch.zeros(EMB_DIM)
print(f"✅ 임베딩 로드: {len(text2emb):,}개  |  dim={EMB_DIM}")


def stt_frames_to_segments(df_stt):
    rows = df_stt.sort("frame_num").to_dicts()
    segments = []
    cur_text = None
    cur_start_frame = None
    prev_frame = None

    for r in rows:
        t = r["stt_text"]
        f = int(r["frame_num"])

        if t != cur_text:
            if cur_text is not None and cur_text != "":
                segments.append({
                    "start": cur_start_frame / FPS,
                    "end": (prev_frame + 1) / FPS,
                    "text": cur_text.strip(),
                })
            cur_text = t
            cur_start_frame = f
        prev_frame = f

    if cur_text is not None and cur_text != "":
        segments.append({
            "start": cur_start_frame / FPS,
            "end": (prev_frame + 1) / FPS,
            "text": cur_text.strip(),
        })
    return segments


def load_one(args):
    channel, path = args
    with open(path, "r") as f:
        data = json.load(f)

    instances = data.get("instances", [])
    duration = data.get("duration", 0.1)
    if instances:
        duration = max(duration, max(inst["end_sec"] for inst in instances))

    video_name = data.get("video", "")

    inst_list = []
    for inst in instances:
        gx = int(np.clip(round(inst["grid_x"]), 0, GRID_W - 1))
        gy = int(np.clip(round(inst["grid_y"]), 0, GRID_H - 1))
        gw = int(np.clip(round(inst["grid_w"]), 1, GRID_W))
        gh = int(np.clip(round(inst["grid_h"]), 1, GRID_H))

        inst_list.append({
            "text": inst["text"],
            "text_len": len(inst["text"]),
            "start": inst["start_sec"],
            "end": inst["end_sec"],
            "x": gx, "y": gy, "w": gw, "h": gh,
        })

    # STT 로드
    fname = os.path.basename(path)[:-5]
    stt_path = os.path.join(STT_DIR, channel, fname + ".parquet")
    stt_segments = []
    if os.path.exists(stt_path):
        try:
            df_stt = pl.read_parquet(stt_path, glob=False)
            stt_segments = stt_frames_to_segments(df_stt)
        except:
            pass

    return {
        "channel": channel,
        "video_name": video_name,
        "instances": inst_list,
        "stt_segments": stt_segments,
        "duration": duration,
    }


# ── 파일 목록 수집 ──
json_paths = []
for channel in sorted(os.listdir(POS_DIR)):
    ch_dir = os.path.join(POS_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for fname in sorted(os.listdir(ch_dir)):
        if fname.endswith(".json"):
            json_paths.append((channel, os.path.join(ch_dir, fname)))

print(f"📁 파일 수: {len(json_paths):,}개")

# ── 병렬 로드 ──
channel_set = set()
samples = []

with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(load_one, args): args for args in json_paths}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="로드"):
        result = fut.result()
        channel_set.add(result["channel"])
        samples.append(result)

channels = sorted(channel_set)
channel2id = {ch: i for i, ch in enumerate(channels)}

# ── 통계 ──
n_with_inst = sum(1 for s in samples if len(s["instances"]) > 0)
n_without_inst = sum(1 for s in samples if len(s["instances"]) == 0)
stt_counts = np.array([len(s["stt_segments"]) for s in samples])
inst_counts = np.array([len(s["instances"]) for s in samples])

print(f"\n📊 영상별 STT 세그먼트 수")
print(f"  mean: {stt_counts.mean():.1f}  median: {np.median(stt_counts):.0f}  "
      f"p99: {np.percentile(stt_counts, 99):.0f}  max: {stt_counts.max()}")

# ── train/eval 분리 ──
rng = random.Random(SEED)
by_channel = {}
for s in samples:
    if s["channel"] not in by_channel:
        by_channel[s["channel"]] = []
    by_channel[s["channel"]].append(s)

train_samples, eval_samples = [], []
for ch, ch_samples in by_channel.items():
    rng.shuffle(ch_samples)
    n_eval = min(EVAL_PER_CHANNEL, len(ch_samples))
    eval_samples.extend(ch_samples[:n_eval])
    train_samples.extend(ch_samples[n_eval:])

print(f"\n✅ 영상: {len(samples):,}개 (텔롭 있음: {n_with_inst:,}  없음: {n_without_inst:,})  |  채널: {len(channels)}개")
print(f"✅ train: {len(train_samples):,}  |  eval: {len(eval_samples):,}")
print(f"📊 인스턴스 수 — mean: {inst_counts.mean():.0f}  p99: {np.percentile(inst_counts, 99):.0f}  max: {inst_counts.max()}")

✅ 임베딩 로드: 2,167,019개  |  dim=1024
📁 파일 수: 66,400개


로드: 100%|██████████| 66400/66400 [00:09<00:00, 6921.23it/s]



📊 영상별 STT 세그먼트 수
  mean: 12.8  median: 10  p99: 51  max: 151

✅ 영상: 66,400개 (텔롭 있음: 59,876  없음: 6,524)  |  채널: 664개
✅ train: 63,080  |  eval: 3,320
📊 인스턴스 수 — mean: 56  p99: 403  max: 4251


In [ ]:
# ── 10개 채널만 샘플링 ──
rng_ch = random.Random(42)
sampled_channels = rng_ch.sample(channels, 67)
sampled_set = set(sampled_channels)

train_samples = [s for s in train_samples if s["channel"] in sampled_set]
eval_samples = [s for s in eval_samples if s["channel"] in sampled_set]
channels = sampled_channels
channel2id = {ch: i for i, ch in enumerate(channels)}

print(f"\n🔬 샘플링: {len(channels)}개 채널")
print(f"   train: {len(train_samples):,}  |  eval: {len(eval_samples):,}")

In [2]:
# %% 셀 2: Dataset + DataLoader (ViT Patch Mask + slot attention)
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

BATCH_SIZE = 8
NUM_WORKERS_DL = 8
MAX_FRAMES = 2000
MAX_ACTIVE_PER_FRAME = 150
MAX_TEXT_LEN = 200
SLOT_DIM = 5  # text_len, ch_sim, vid_sim, stt_sim, has_stt
TIME_DIM = 3  # t_norm, sin(2πt), cos(2πt)
PATCH_SIZE = 8
N_PATCHES_X = GRID_W // PATCH_SIZE  # 10
N_PATCHES_Y = GRID_H // PATCH_SIZE  # 10
N_PATCHES = N_PATCHES_X * N_PATCHES_Y  # 100
POS_WEIGHT = 12.2


class FrameMaskViTDataset(Dataset):
    def __init__(self, samples, channel2id, text2emb):
        self.samples = [s for s in samples if len(s["instances"]) > 0]
        self.channel2id = channel2id
        self.text2emb = text2emb

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        channel_id = self.channel2id[s["channel"]]
        instances = s["instances"]
        duration = max(s["duration"], 0.1)

        n_frames = max(1, min(int(duration * FPS) + 1, MAX_FRAMES))

        # ── 인스턴스 배열 ──
        n_inst = len(instances)
        inst_starts = np.array([inst["start"] for inst in instances])
        inst_ends   = np.array([inst["end"]   for inst in instances])
        inst_tlens  = np.array([inst["text_len"] for inst in instances])
        inst_cx = np.array([inst["x"] for inst in instances])
        inst_cy = np.array([inst["y"] for inst in instances])
        inst_w  = np.array([inst["w"] for inst in instances])
        inst_h  = np.array([inst["h"] for inst in instances])

        # ── 인스턴스별 유사도 사전 계산 ──
        channel_emb = self.text2emb.get(s["channel"], ZERO_EMB)
        video_emb   = self.text2emb.get(s["video_name"], ZERO_EMB)

        inst_embs = torch.stack([
            self.text2emb.get(inst["text"], ZERO_EMB) for inst in instances
        ])
        ch_sims  = F.cosine_similarity(inst_embs, channel_emb.unsqueeze(0), dim=-1).numpy()
        vid_sims = F.cosine_similarity(inst_embs, video_emb.unsqueeze(0), dim=-1).numpy()

        stt_sims = np.zeros(n_inst, dtype=np.float32)
        has_stts = np.zeros(n_inst, dtype=np.float32)

        stt_segments = s["stt_segments"]
        if len(stt_segments) > 0:
            stt_starts = np.array([seg["start"] for seg in stt_segments])
            stt_ends   = np.array([seg["end"]   for seg in stt_segments])
            stt_embs   = torch.stack([
                self.text2emb.get(seg["text"], ZERO_EMB) for seg in stt_segments
            ])

            for i in range(n_inst):
                mid = (inst_starts[i] + inst_ends[i]) / 2
                stt_active = (stt_starts <= mid) & (stt_ends > mid)
                stt_active_idx = np.where(stt_active)[0]
                if len(stt_active_idx) > 0:
                    stt_sims[i] = F.cosine_similarity(
                        inst_embs[i].unsqueeze(0),
                        stt_embs[stt_active_idx[0]].unsqueeze(0),
                    ).item()
                    has_stts[i] = 1.0

        # ── 프레임별 활성 텔롭 매트릭스 ──
        times = np.arange(n_frames, dtype=np.float32) / FPS
        active_matrix = (
            (inst_starts[None, :] <= times[:, None] + 0.05) &
            (inst_ends[None, :]   >  times[:, None])
        )

        # ── 프레임별 텔롭 슬롯 (150 × 5d) ──
        telop_feats = np.zeros((n_frames, MAX_ACTIVE_PER_FRAME, SLOT_DIM), dtype=np.float32)
        telop_mask  = np.zeros((n_frames, MAX_ACTIVE_PER_FRAME), dtype=np.bool_)

        for fi in range(n_frames):
            active_idx = np.where(active_matrix[fi])[0]
            if len(active_idx) > 0:
                sorted_order = np.argsort(inst_tlens[active_idx])[::-1][:MAX_ACTIVE_PER_FRAME]
                sorted_idx = active_idx[sorted_order]

                for si, ai in enumerate(sorted_idx):
                    telop_feats[fi, si, 0] = inst_tlens[ai] / MAX_TEXT_LEN
                    telop_feats[fi, si, 1] = ch_sims[ai]
                    telop_feats[fi, si, 2] = vid_sims[ai]
                    telop_feats[fi, si, 3] = stt_sims[ai]
                    telop_feats[fi, si, 4] = has_stts[ai]
                    telop_mask[fi, si] = True

        # ── 시간 feature (3d) ──
        time_feats = np.zeros((n_frames, TIME_DIM), dtype=np.float32)
        t_norm = times / max(duration, 1.0)
        time_feats[:, 0] = t_norm
        time_feats[:, 1] = np.sin(2 * np.pi * t_norm)
        time_feats[:, 2] = np.cos(2 * np.pi * t_norm)

        # ── GT mask 생성 ──
        masks = np.zeros((n_frames, GRID_H, GRID_W), dtype=np.float32)

        for j in range(n_inst):
            cx, cy = int(inst_cx[j]), int(inst_cy[j])
            w, h   = int(inst_w[j]),  int(inst_h[j])
            x0 = max(0, cx - w // 2)
            y0 = max(0, cy - h // 2)
            x1 = min(GRID_W, x0 + w)
            y1 = min(GRID_H, y0 + h)
            if x1 <= x0 or y1 <= y0:
                continue
            active_frames = np.where(active_matrix[:, j])[0]
            if len(active_frames) > 0:
                masks[active_frames[:, None, None],
                      np.arange(y0, y1)[None, :, None],
                      np.arange(x0, x1)[None, None, :]] = 1.0

        return {
            "channel_id":  torch.tensor(channel_id, dtype=torch.long),
            "telop_feats": torch.from_numpy(telop_feats),
            "telop_mask":  torch.from_numpy(telop_mask),
            "time_feats":  torch.from_numpy(time_feats),
            "masks":       torch.from_numpy(masks),
            "n_frames":    n_frames,
        }


def collate_fn(batch):
    max_T = max(b["n_frames"] for b in batch)
    B = len(batch)

    channel_ids = torch.stack([b["channel_id"] for b in batch])
    telop_feats = torch.zeros(B, max_T, MAX_ACTIVE_PER_FRAME, SLOT_DIM)
    telop_mask  = torch.zeros(B, max_T, MAX_ACTIVE_PER_FRAME, dtype=torch.bool)
    time_feats  = torch.zeros(B, max_T, TIME_DIM)
    masks       = torch.zeros(B, max_T, GRID_H, GRID_W)
    frame_mask  = torch.zeros(B, max_T, dtype=torch.bool)

    for i, b in enumerate(batch):
        T = b["n_frames"]
        telop_feats[i, :T] = b["telop_feats"]
        telop_mask[i, :T]  = b["telop_mask"]
        time_feats[i, :T]  = b["time_feats"]
        masks[i, :T]       = b["masks"]
        frame_mask[i, :T]  = True

    return {
        "channel_ids": channel_ids,
        "telop_feats": telop_feats,
        "telop_mask":  telop_mask,
        "time_feats":  time_feats,
        "masks":       masks,
        "frame_mask":  frame_mask,
    }


train_ds = FrameMaskViTDataset(train_samples, channel2id, text2emb)
eval_ds  = FrameMaskViTDataset(eval_samples, channel2id, text2emb)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS_DL, pin_memory=True,
    collate_fn=collate_fn, drop_last=True,
    persistent_workers=True,
)
eval_loader = DataLoader(
    eval_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS_DL, pin_memory=True,
    collate_fn=collate_fn,
    persistent_workers=True,
)

print(f"✅ Dataset: train={len(train_ds):,}  eval={len(eval_ds):,}")
print(f"   SLOT_DIM={SLOT_DIM}  MAX_ACTIVE_PER_FRAME={MAX_ACTIVE_PER_FRAME}")
print(f"   N_PATCHES={N_PATCHES}  PATCH_SIZE={PATCH_SIZE}")
print(f"   MAX_FRAMES={MAX_FRAMES}  POS_WEIGHT={POS_WEIGHT}")

batch = next(iter(train_loader))
print(f"\n✅ 배치 확인")
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape}")

valid_masks = batch["masks"][batch["frame_mask"]]
pos_ratio = valid_masks.mean().item()
print(f"\n📊 배치 mask positive 비율: {pos_ratio:.4f} (weight ≈ {1/max(pos_ratio,1e-6):.1f}:1)")

✅ Dataset: train=56,869  eval=3,007
   SLOT_DIM=5  MAX_ACTIVE_PER_FRAME=150
   N_PATCHES=100  PATCH_SIZE=8
   MAX_FRAMES=2000  POS_WEIGHT=12.2

✅ 배치 확인
  channel_ids: torch.Size([8])
  telop_feats: torch.Size([8, 1037, 150, 5])
  telop_mask: torch.Size([8, 1037, 150])
  time_feats: torch.Size([8, 1037, 3])
  masks: torch.Size([8, 1037, 80, 80])
  frame_mask: torch.Size([8, 1037])

📊 배치 mask positive 비율: 0.1204 (weight ≈ 8.3:1)


In [3]:
# %% 셀 3: 모델 정의 (ViT Patch Mask + multi-pooling)
D_MODEL = 256
N_HEADS = 8
N_LAYERS_TEMPORAL = 4
N_LAYERS_SPATIAL = 4
D_FF = 512
DROPOUT = 0.1
INTRA_CHUNK = 512
SPATIAL_CHUNK = 128


class IntraFrameModule(nn.Module):
    def __init__(self, d_model=D_MODEL):
        super().__init__()

        self.slot_proj = nn.Sequential(
            nn.Linear(SLOT_DIM, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

        self.pool_query = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        self.combine = nn.Sequential(
            nn.Linear(d_model * 3 + 1, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(self, slot_feats, slot_mask):
        N, K, _ = slot_feats.shape
        device = slot_feats.device
        dtype = slot_feats.dtype

        tokens = self.slot_proj(slot_feats)
        d = tokens.shape[-1]

        has_any = slot_mask.any(dim=1)
        pooled = torch.zeros(N, d, device=device, dtype=dtype)

        if has_any.any():
            valid_idx = has_any.nonzero(as_tuple=True)[0]
            valid_tokens = tokens[valid_idx]
            valid_mask = slot_mask[valid_idx]
            valid_pad = ~valid_mask
            V = valid_tokens.shape[0]

            count = valid_mask.sum(dim=1, keepdim=True).float()
            count_norm = count / MAX_ACTIVE_PER_FRAME

            masked_tokens = valid_tokens.masked_fill(valid_pad.unsqueeze(-1), 0.0)
            mean_pool = masked_tokens.sum(dim=1) / count.clamp(min=1)

            masked_for_max = valid_tokens.masked_fill(valid_pad.unsqueeze(-1), float("-inf"))
            max_pool = masked_for_max.max(dim=1).values

            q = self.pool_query.expand(V, -1, -1)
            a = torch.bmm(q, valid_tokens.transpose(1, 2)) / (d ** 0.5)
            a = a.masked_fill(valid_pad.unsqueeze(1), float("-inf"))
            a = F.softmax(a, dim=-1)
            attn_pool = torch.bmm(a, valid_tokens).squeeze(1)

            combined = torch.cat([attn_pool, mean_pool, max_pool, count_norm], dim=-1)
            pooled[valid_idx] = self.norm(self.combine(combined)).to(dtype)

        return pooled


class ViTPatchMaskModel(nn.Module):
    def __init__(self, n_channels, d_model=D_MODEL, n_heads=N_HEADS,
                 n_layers_temporal=N_LAYERS_TEMPORAL,
                 n_layers_spatial=N_LAYERS_SPATIAL,
                 d_ff=D_FF, dropout=DROPOUT):
        super().__init__()

        self.intra_frame = IntraFrameModule(d_model)

        self.channel_emb = nn.Embedding(n_channels, d_model)

        self.time_proj = nn.Sequential(
            nn.Linear(TIME_DIM, 64),
            nn.GELU(),
            nn.Linear(64, d_model),
        )

        self.temporal_norm = nn.LayerNorm(d_model)

        temporal_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.temporal_transformer = nn.TransformerEncoder(
            temporal_layer, num_layers=n_layers_temporal,
            enable_nested_tensor=False,
        )

        self.patch_pos_emb = nn.Parameter(
            torch.randn(1, N_PATCHES, d_model) * 0.02
        )

        self.spatial_norm = nn.LayerNorm(d_model)

        spatial_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.spatial_transformer = nn.TransformerEncoder(
            spatial_layer, num_layers=n_layers_spatial,
            enable_nested_tensor=False,
        )

        self.patch_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, PATCH_SIZE * PATCH_SIZE),
        )

    def forward(self, channel_ids, telop_feats, telop_mask, time_feats, frame_mask):
        B, T, K, _ = telop_feats.shape
        device = telop_feats.device
        dtype = telop_feats.dtype
        BT = B * T

        slot_flat = telop_feats.reshape(BT, K, SLOT_DIM)
        smask_flat = telop_mask.reshape(BT, K)

        frame_tokens_list = []
        for start in range(0, BT, INTRA_CHUNK):
            end = min(start + INTRA_CHUNK, BT)
            chunk_out = self.intra_frame(slot_flat[start:end], smask_flat[start:end])
            frame_tokens_list.append(chunk_out)

        frame_tokens = torch.cat(frame_tokens_list, dim=0).reshape(B, T, -1)

        ch   = self.channel_emb(channel_ids).unsqueeze(1).expand(-1, T, -1)
        time = self.time_proj(time_feats)
        tokens = self.temporal_norm(frame_tokens + ch + time)

        temporal_out = self.temporal_transformer(
            tokens, src_key_padding_mask=~frame_mask
        )

        temporal_flat = temporal_out.reshape(BT, -1)
        valid_flat    = frame_mask.reshape(BT)
        valid_idx     = valid_flat.nonzero(as_tuple=True)[0]
        n_valid       = valid_idx.shape[0]

        all_logits = torch.zeros(BT, GRID_H, GRID_W, device=device, dtype=dtype)

        if n_valid > 0:
            valid_ctx = temporal_flat[valid_idx]
            queries = valid_ctx.unsqueeze(1) + self.patch_pos_emb
            queries = self.spatial_norm(queries)

            chunk_list = []
            for start in range(0, n_valid, SPATIAL_CHUNK):
                end = min(start + SPATIAL_CHUNK, n_valid)
                chunk_q = queries[start:end]
                spatial_out = self.spatial_transformer(chunk_q)
                patch_logits = self.patch_head(spatial_out)
                chunk_list.append(patch_logits)

            all_patch = torch.cat(chunk_list, dim=0)

            mask_logits = (
                all_patch
                .reshape(-1, N_PATCHES_Y, N_PATCHES_X, PATCH_SIZE, PATCH_SIZE)
                .permute(0, 1, 3, 2, 4)
                .contiguous()
                .reshape(-1, GRID_H, GRID_W)
            )

            all_logits[valid_idx] = mask_logits.to(all_logits.dtype)

        return all_logits.reshape(B, T, GRID_H, GRID_W)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ViTPatchMaskModel(n_channels=len(channels)).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"🖥️  Device: {device}")
print(f"📐 파라미터: {n_params:,}")
print(f"   temporal: {N_LAYERS_TEMPORAL} layers  spatial: {N_LAYERS_SPATIAL} layers")
print(f"   d_model={D_MODEL}  n_heads={N_HEADS}  d_ff={D_FF}")
print(f"   INTRA_CHUNK={INTRA_CHUNK}  SPATIAL_CHUNK={SPATIAL_CHUNK}")

🖥️  Device: cuda
📐 파라미터: 4,843,584
   temporal: 4 layers  spatial: 4 layers
   d_model=256  n_heads=8  d_ff=512
   INTRA_CHUNK=512  SPATIAL_CHUNK=128


In [ ]:
# %% 셀 4: 학습 (ViT Patch Mask + multi-pooling + Focal Loss)
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS = 50
LR = 1e-4
FOCAL_ALPHA = POS_WEIGHT / (1 + POS_WEIGHT)  # positive class weight (≈0.924)
FOCAL_GAMMA = 2.0
SAVE_DIR = "./model/8_layout_vit_patch_mask_focal"
os.makedirs(SAVE_DIR, exist_ok=True)


def focal_loss(logits, targets, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA):
    """
    Binary Focal Loss with alpha balancing.
    alpha: positive class weight (pos_weight를 alpha로 변환)
    gamma: focusing parameter (높을수록 쉬운 example 무시)
    """
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    probs = torch.sigmoid(logits)

    # p_t: 정답 클래스에 대한 확률
    p_t = probs * targets + (1 - probs) * (1 - targets)

    # alpha_t: 정답 클래스에 따른 가중치
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)

    # focal weight: 쉬운 example일수록 (1-p_t)^gamma이 작아져서 loss 감소
    focal_weight = alpha_t * (1 - p_t) ** gamma

    return (focal_weight * bce).mean()


@torch.no_grad()
def compute_mask_metrics(pred_logits, gt_masks, frame_mask,
                         thresholds=(0.2, 0.3, 0.4, 0.5, 0.6, 0.7)):
    if not frame_mask.any():
        return None

    pred_prob = torch.sigmoid(pred_logits[frame_mask])
    gt_bool   = gt_masks[frame_mask].bool()

    best_f1, best_th = 0.0, 0.5
    result_at_05 = {}

    for th in thresholds:
        pred_bin = pred_prob > th
        tp = (pred_bin & gt_bool).sum().item()
        fp = (pred_bin & ~gt_bool).sum().item()
        fn = (~pred_bin & gt_bool).sum().item()

        p = tp / (tp + fp + 1e-8)
        r = tp / (tp + fn + 1e-8)
        f1 = 2 * p * r / (p + r + 1e-8)

        if th == 0.5:
            result_at_05 = {"P": p, "R": r, "F1": f1}

        if f1 > best_f1:
            best_f1 = f1
            best_th = th

    result_at_05["bestF1"] = best_f1
    result_at_05["bestTh"] = best_th
    return result_at_05


optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_eval_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum, train_batches = 0.0, 0

    for batch in tqdm(train_loader, desc=f"[{epoch}/{EPOCHS}] train", leave=False):
        channel_ids = batch["channel_ids"].to(device)
        telop_feats = batch["telop_feats"].to(device)
        telop_mask  = batch["telop_mask"].to(device)
        time_feats  = batch["time_feats"].to(device)
        gt_masks    = batch["masks"].to(device)
        frame_mask  = batch["frame_mask"].to(device)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            pred_logits = model(channel_ids, telop_feats, telop_mask,
                                time_feats, frame_mask)

            if frame_mask.any():
                loss = focal_loss(
                    pred_logits[frame_mask], gt_masks[frame_mask],
                )
            else:
                loss = torch.tensor(0.0, device=device, requires_grad=True)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss_sum += loss.item()
        train_batches += 1

    model.eval()
    eval_loss_sum, eval_batches = 0.0, 0
    all_metrics = []

    with torch.no_grad():
        for batch in eval_loader:
            channel_ids = batch["channel_ids"].to(device)
            telop_feats = batch["telop_feats"].to(device)
            telop_mask  = batch["telop_mask"].to(device)
            time_feats  = batch["time_feats"].to(device)
            gt_masks    = batch["masks"].to(device)
            frame_mask  = batch["frame_mask"].to(device)

            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                pred_logits = model(channel_ids, telop_feats, telop_mask,
                                    time_feats, frame_mask)

                if frame_mask.any():
                    loss = focal_loss(
                        pred_logits[frame_mask], gt_masks[frame_mask],
                    )
                else:
                    loss = torch.tensor(0.0, device=device)

            eval_loss_sum += loss.item()
            eval_batches += 1

            m = compute_mask_metrics(pred_logits, gt_masks, frame_mask)
            if m is not None:
                all_metrics.append(m)

    scheduler.step()

    train_loss = train_loss_sum / max(train_batches, 1)
    eval_loss  = eval_loss_sum  / max(eval_batches, 1)
    lr_now = optimizer.param_groups[0]["lr"]

    if all_metrics:
        avg_m  = {k: np.mean([m[k] for m in all_metrics])
                  for k in all_metrics[0] if k != "bestTh"}
        avg_th = np.mean([m["bestTh"] for m in all_metrics])
    else:
        avg_m  = {"P": 0, "R": 0, "F1": 0, "bestF1": 0}
        avg_th = 0.5

    print(
        f"[{epoch:>3}/{EPOCHS}]  "
        f"train={train_loss:.4f}  eval={eval_loss:.4f}  "
        f"P={avg_m['P']:.3f}  R={avg_m['R']:.3f}  "
        f"F1={avg_m['F1']:.3f}  bestF1={avg_m['bestF1']:.3f}@{avg_th:.2f}  "
        f"lr={lr_now:.2e}"
    )

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "eval_loss": eval_loss,
        "eval_metrics": avg_m,
        "channel2id": channel2id,
    }, os.path.join(SAVE_DIR, f"epoch_{epoch:03d}.pt"))


    if eval_loss < best_eval_loss:
        best_eval_loss = eval_loss
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "eval_loss": eval_loss,
            "eval_metrics": avg_m,
            "channel2id": channel2id,
        }, os.path.join(SAVE_DIR, "best.pt"))

        print(f"   💾 best 갱신 (eval_loss={eval_loss:.4f})")

print(f"\n✅ 완료. Best eval_loss={best_eval_loss:.4f}")

[  1/50]  train=0.0173  eval=0.0145  P=0.222  R=0.759  F1=0.336  bestF1=0.429@0.59  lr=9.99e-05
   💾 best 갱신 (eval_loss=0.0145)


[  2/50]  train=0.0144  eval=0.0132  P=0.240  R=0.824  F1=0.366  bestF1=0.475@0.60  lr=9.96e-05
   💾 best 갱신 (eval_loss=0.0132)


[  3/50]  train=0.0133  eval=0.0123  P=0.277  R=0.816  F1=0.408  bestF1=0.506@0.63  lr=9.91e-05
   💾 best 갱신 (eval_loss=0.0123)


[  4/50]  train=0.0125  eval=0.0116  P=0.285  R=0.841  F1=0.420  bestF1=0.522@0.61  lr=9.84e-05
   💾 best 갱신 (eval_loss=0.0116)


[  5/50]  train=0.0119  eval=0.0111  P=0.296  R=0.849  F1=0.433  bestF1=0.538@0.62  lr=9.76e-05
   💾 best 갱신 (eval_loss=0.0111)


...

[ 24/50]  train=0.0079  eval=0.0100  P=0.390  R=0.857  F1=0.528  bestF1=0.627@0.66  lr=5.31e-05


[ 25/50]  train=0.0078  eval=0.0102  P=0.396  R=0.852  F1=0.534  bestF1=0.626@0.66  lr=5.00e-05


[ 26/50]  train=0.0076  eval=0.0103  P=0.399  R=0.847  F1=0.535  bestF1=0.624@0.65  lr=4.69e-05


[ 27/50]  train=0.0075  eval=0.0104  P=0.386  R=0.858  F1=0.525  bestF1=0.627@0.67  lr=4.37e-05


[28/50] train:  70%|██████▉   | 4970/7108 [14:51<05:56,  5.99it/s]